# AdaptivePointNet — Dual Attention Blocks for 3D Point Cloud Classification

**Novel Contribution:** Two attention mechanisms are added to the canonical PointNet backbone:

| Block | Position | Purpose |
|-------|----------|---------|
| **Local Channel Attention Block (LCAB)** | After first two `conv_bn(64)` layers | Squeeze-and-excitation channel recalibration across 64 feature channels per point |
| **Global Self-Attention Block (GSAB)** | After final `conv_bn(1024)` and before `GlobalMaxPool` | Multi-head self-attention across all N points to capture long-range inter-point dependencies |

**Architecture flow:**
```
Input (N,3)
  → T-Net 1 (3×3)
  → conv_bn(64) → conv_bn(64)
  → ★ LCAB: Local Channel Attention        ← NEW
  → T-Net 2 (64×64)
  → conv_bn(64) → conv_bn(128) → conv_bn(1024)
  → ★ GSAB: Global Self-Attention          ← NEW
  → GlobalMaxPooling
  → dense_bn(512) → dense_bn(256) → Dropout(0.3)
  → Dense(10, softmax)
```

**References:**
- Qi et al., PointNet, CVPR 2017 — https://arxiv.org/abs/1612.00593
- Hu et al., Squeeze-and-Excitation Networks, CVPR 2018 (LCAB inspiration)
- Vaswani et al., Attention Is All You Need, NeurIPS 2017 (GSAB inspiration)
- Zhao et al., Point Transformer, ICCV 2021 (point-cloud self-attention)


## 1. Import Libraries

In [ ]:
import os
import glob
import tqdm
import pickle

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
%matplotlib inline

tf.random.set_seed(42)
print("TensorFlow version:", tf.__version__)

In [ ]:
# Install trimesh — handles OFF (Object File Format) mesh files
!pip -q install trimesh
import trimesh
print("trimesh imported successfully")

## 2. Configuration

In [ ]:
DATA_DIR     = 'ModelNet10/'
PARSED_DIR   = '/kaggle/input/parsed-data/'
WEIGHTS_PATH = 'adaptivepointnet_attention_weights.hdf5'

NUM_POINTS   = 2048    # points sampled from each mesh
NUM_CLASSES  = 10      # ModelNet10 categories
BATCH_SIZE   = 32

print(f"Config — NUM_POINTS={NUM_POINTS}, NUM_CLASSES={NUM_CLASSES}, BATCH_SIZE={BATCH_SIZE}")

## 3. Visualise Raw Mesh Data

In [ ]:
# Load two sample meshes and visualise their sampled point clouds
mesh_toilet  = trimesh.load(os.path.join(DATA_DIR, 'toilet/train/toilet_0001.off'))
mesh_bathtub = trimesh.load(os.path.join(DATA_DIR, 'bathtub/train/bathtub_0010.off'))

pts_toilet  = mesh_toilet.sample(2048)
pts_bathtub = mesh_bathtub.sample(2048)

fig = plt.figure(figsize=(12, 5))
ax1 = fig.add_subplot(121, projection='3d')
ax1.scatter(pts_toilet[:, 0], pts_toilet[:, 1], pts_toilet[:, 2], s=1, c=pts_toilet[:, 2], cmap='plasma')
ax1.set_title('Toilet point cloud (2048 pts)')
ax1.set_axis_off()

ax2 = fig.add_subplot(122, projection='3d')
ax2.scatter(pts_bathtub[:, 0], pts_bathtub[:, 1], pts_bathtub[:, 2], s=1, c=pts_bathtub[:, 2], cmap='viridis')
ax2.set_title('Bathtub point cloud (2048 pts)')
ax2.set_axis_off()

plt.suptitle('Uniform surface sampling from OFF meshes', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Dataset Parsing

In [ ]:
def parse_dataset(num_points=2048):
    """
    Walk the ModelNet10 directory, uniformly sample num_points from each
    OFF mesh surface, and return train/test numpy arrays with labels.
    """
    train_points, train_labels = [], []
    test_points,  test_labels  = [], []
    class_map = {}

    folders = os.listdir(DATA_DIR)
    for i, folder in tqdm.tqdm(enumerate(folders)):
        print('Processing class: {}'.format(folder))
        class_map[i] = folder
        for f in glob.glob(os.path.join(DATA_DIR, folder, 'train/*')):
            train_points.append(trimesh.load(f).sample(num_points))
            train_labels.append(i)
        for f in glob.glob(os.path.join(DATA_DIR, folder, 'test/*')):
            test_points.append(trimesh.load(f).sample(num_points))
            test_labels.append(i)

    return (np.array(train_points), np.array(train_labels),
            np.array(test_points),  np.array(test_labels), class_map)

In [ ]:
# Load pre-parsed numpy arrays (faster); parse from scratch if not found
try:
    train_points = np.load(PARSED_DIR + '3d_train_points_MNet10.npy')
    train_labels = np.load(PARSED_DIR + '3d_train_labels_MNet10.npy')
    test_points  = np.load(PARSED_DIR + '3d_test_points_MNet10.npy')
    test_labels  = np.load(PARSED_DIR + '3d_test_labels_MNet10.npy')
    with open(PARSED_DIR + 'class_map_dict.pkl', 'rb') as f:
        class_map = pickle.load(f)
    print('Loaded pre-parsed data.')
except FileNotFoundError:
    print('Parsing from scratch …')
    train_points, train_labels, test_points, test_labels, class_map = parse_dataset(NUM_POINTS)
    np.save('3d_train_points_MNet10.npy', train_points)
    np.save('3d_train_labels_MNet10.npy', train_labels)
    np.save('3d_test_points_MNet10.npy',  test_points)
    np.save('3d_test_labels_MNet10.npy',  test_labels)
    with open('class_map_dict.pkl', 'wb') as f:
        pickle.dump(class_map, f)

print(f'Train: {train_points.shape}  |  Test: {test_points.shape}')
print(f'Classes: {class_map}')

## 5. Data Augmentation & tf.data Pipelines

In [ ]:
def augment(points, label):
    """
    Two augmentations applied during training only:
      - Jitter  : add uniform noise in [-0.005, 0.005] per coordinate
      - Shuffle : randomly permute point order (PointNet is order-invariant)
    """
    points += tf.random.uniform(points.shape, -0.005, 0.005, dtype=tf.float64)
    points  = tf.random.shuffle(points)
    return points, label

train_dataset = tf.data.Dataset.from_tensor_slices((train_points, train_labels))
test_dataset  = tf.data.Dataset.from_tensor_slices((test_points,  test_labels))

train_dataset = train_dataset.shuffle(len(train_points)).map(augment).batch(BATCH_SIZE)
test_dataset  = test_dataset.batch(BATCH_SIZE)

print("Datasets ready.")

## 6. Model Building Blocks

### 6a. Standard Blocks: `conv_bn` and `dense_bn`

In [ ]:
def conv_bn(x, filters):
    """Shared 1-D conv (kernel=1) → BatchNorm → ReLU.
    kernel_size=1 ensures each point is processed independently,
    preserving permutation equivariance."""
    x = layers.Conv1D(filters, kernel_size=1, padding='valid')(x)
    x = layers.BatchNormalization(momentum=0.0)(x)
    return layers.Activation('relu')(x)

def dense_bn(x, filters):
    """Dense → BatchNorm → ReLU."""
    x = layers.Dense(filters)(x)
    x = layers.BatchNormalization(momentum=0.0)(x)
    return layers.Activation('relu')(x)

### 6b. Orthogonal Regulariser & T-Net

In [ ]:
class OrthogonalRegularizer(keras.regularizers.Regularizer):
    """
    Penalises T-Net output matrix T from deviating from orthogonality:
        loss = l2reg * ||T T^T - I||_F^2
    Keeps the learned transform close to a rotation, preventing information loss.
    """
    def __init__(self, num_features, l2reg=0.001):
        self.num_features = num_features
        self.l2reg        = l2reg
        self.eye          = tf.eye(num_features)

    def __call__(self, x):
        x   = tf.reshape(x, (-1, self.num_features, self.num_features))
        xxt = tf.tensordot(x, x, axes=(2, 2))
        xxt = tf.reshape(xxt, (-1, self.num_features, self.num_features))
        return tf.reduce_sum(self.l2reg * tf.square(xxt - self.eye))

    def get_config(self):
        return {'num_features': self.num_features, 'l2reg': self.l2reg}


def tnet(inputs, num_features):
    """
    T-Net: predicts a (num_features x num_features) affine transform
    and applies it to the input to align points/features to a
    canonical space before feature extraction.

    Architecture: shared MLP(64,128,1024) → GlobalMaxPool
                  → Dense(512) → Dense(256) → Dense(num_features^2)
    """
    bias   = keras.initializers.Constant(np.eye(num_features).flatten())
    reg    = OrthogonalRegularizer(num_features)

    x = conv_bn(inputs, 64)
    x = conv_bn(x, 128)
    x = conv_bn(x, 1024)
    x = layers.GlobalMaxPooling1D()(x)
    x = dense_bn(x, 512)
    x = dense_bn(x, 256)
    x = layers.Dense(
        num_features * num_features,
        kernel_initializer='Identity',
        bias_initializer=bias,
        activity_regularizer=reg,
    )(x)
    feat_T = layers.Reshape((num_features, num_features))(x)
    return layers.Dot(axes=(2, 1))([inputs, feat_T])

### 6c. ★ Attention Block 1: Local Channel Attention Block (LCAB)

**Where it sits:** After the first two `conv_bn(64)` layers, before T-Net 2.

**What it does:** Channel-wise Squeeze-and-Excitation adapted for point cloud sequences.  
Each of the 64 feature channels gets a learned scalar weight in [0,1] based on the  
global average *and* global max of that channel across all N points.  
Channels encoding geometrically important features (e.g., surface normals, curvature)  
are amplified; noisy or redundant channels are suppressed.

```
Input (B, N, 64)
  Squeeze: global_avg_pool → (B, 64)
           global_max_pool → (B, 64)
           concat          → (B, 128)
  Excite:  Dense(8, relu) → Dense(64, sigmoid) → gate (B, 64)
  Scale:   reshape gate to (B, 1, 64), multiply with input → (B, N, 64)
  Residual: output = input + scaled               → (B, N, 64)
```


In [ ]:
def local_channel_attention_block(x, reduction_ratio=8, name='lcab'):
    """
    Local Channel Attention Block (LCAB)
    =====================================
    Squeeze-and-Excitation channel recalibration for point cloud features.

    Steps:
      1. Squeeze  : compute global avg-pool AND global max-pool → concat (B, 2C)
      2. Excite   : two-layer MLP gate: Dense(C/r, relu) → Dense(C, sigmoid)
      3. Scale    : element-wise multiply input with gate (broadcast over N)
      4. Residual : add original input for stable gradient flow

    Args:
        x              : (B, N, C) per-point feature map
        reduction_ratio: bottleneck factor r for the excitation MLP
        name           : layer name prefix

    Returns:
        (B, N, C) recalibrated feature map
    """
    C = x.shape[-1]

    # ── Step 1: Squeeze ───────────────────────────────────────────────
    f_avg = layers.GlobalAveragePooling1D(name=f'{name}_gap')(x)    # (B, C)
    f_max = layers.GlobalMaxPooling1D    (name=f'{name}_gmp')(x)    # (B, C)
    f_cat = layers.Concatenate          (name=f'{name}_cat')([f_avg, f_max])  # (B, 2C)

    # ── Step 2: Excite (bottleneck MLP) ──────────────────────────────
    bottleneck = max(1, C // reduction_ratio)
    h    = layers.Dense(bottleneck, activation='relu',
                        use_bias=False, name=f'{name}_fc1')(f_cat)  # (B, C/r)
    gate = layers.Dense(C, activation='sigmoid',
                        use_bias=False, name=f'{name}_fc2')(h)      # (B, C)

    # ── Step 3: Scale (broadcast gate over N) ────────────────────────
    gate  = layers.Reshape((1, C), name=f'{name}_reshape')(gate)    # (B, 1, C)
    x_att = layers.Multiply(name=f'{name}_scale')([x, gate])        # (B, N, C)

    # ── Step 4: Residual ──────────────────────────────────────────────
    x_out = layers.Add       (name=f'{name}_add')  ([x, x_att])
    x_out = layers.Activation('relu', name=f'{name}_relu')(x_out)
    return x_out

print("LCAB defined.")

### 6d. ★ Attention Block 2: Global Self-Attention Block (GSAB)

**Where it sits:** After `conv_bn(1024)`, before `GlobalMaxPooling1D`.

**What it does:** Multi-head self-attention across all N point tokens — a single  
Transformer encoder layer. Each point can attend to all other points, capturing  
long-range geometric relationships that independent per-point MLPs cannot model.  
This enriches the 1024-d descriptors before the irreversible max-pool aggregation.

```
Input (B, N, 1024)
  LayerNorm → MultiHeadAttention(4 heads, key_dim=64) → Dropout → Add  (residual 1)
  LayerNorm → FFN: Dense(4096, relu) → Dense(1024) → Dropout → Add      (residual 2)
Output (B, N, 1024)
```


In [ ]:
def global_self_attention_block(x, num_heads=4, key_dim=64, name='gsab'):
    """
    Global Self-Attention Block (GSAB)
    =====================================
    A single Transformer encoder layer applied across all N point tokens.

    Steps:
      1. Pre-LayerNorm → MultiHeadAttention (Q=K=V=x_norm)
      2. Dropout + residual connection
      3. Pre-LayerNorm → Feed-Forward Network (Dense 4C → Dense C)
      4. Dropout + residual connection

    Args:
        x         : (B, N, C) per-point feature map
        num_heads : number of attention heads (default 4)
        key_dim   : dimensionality per head (default 64)
        name      : layer name prefix

    Returns:
        (B, N, C) context-enriched feature map
    """
    C = x.shape[-1]

    # ── Block 1: Multi-Head Self-Attention ────────────────────────────
    x_norm   = layers.LayerNormalization(epsilon=1e-6, name=f'{name}_ln1')(x)
    attn_out = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=key_dim,
        dropout=0.1,
        name=f'{name}_mhsa',
    )(x_norm, x_norm)                                       # (B, N, C)
    attn_out = layers.Dropout(0.1, name=f'{name}_attn_drop')(attn_out)
    x_res1   = layers.Add   (name=f'{name}_res1')([x, attn_out])  # residual

    # ── Block 2: Feed-Forward Network ────────────────────────────────
    x_norm2  = layers.LayerNormalization(epsilon=1e-6, name=f'{name}_ln2')(x_res1)
    ffn      = layers.Dense(C * 4, activation='relu', name=f'{name}_ffn1')(x_norm2)
    ffn      = layers.Dense(C,                        name=f'{name}_ffn2')(ffn)
    ffn      = layers.Dropout(0.1, name=f'{name}_ffn_drop')(ffn)
    x_out    = layers.Add   (name=f'{name}_res2')([x_res1, ffn])  # residual

    return x_out

print("GSAB defined.")

## 7. Build AdaptivePointNet with Dual Attention

In [ ]:
def build_adaptivepointnet(num_points=NUM_POINTS, num_classes=NUM_CLASSES):
    """
    Full AdaptivePointNet model with two attention blocks.

    Architecture:
        Input (N,3)
        → T-Net 1 (3×3 spatial transform)
        → conv_bn(64) → conv_bn(64)
        → ★ LCAB (Local Channel Attention)
        → T-Net 2 (64×64 feature transform)
        → conv_bn(64) → conv_bn(128) → conv_bn(1024)
        → ★ GSAB (Global Self-Attention)
        → GlobalMaxPooling1D
        → dense_bn(512) → dense_bn(256) → Dropout(0.3)
        → Dense(num_classes, softmax)
    """
    inputs = keras.Input(shape=(num_points, 3), name='point_cloud_input')

    # ── Spatial alignment ────────────────────────────────────────────
    x = tnet(inputs, 3)

    # ── First feature extraction ─────────────────────────────────────
    x = conv_bn(x, 64)
    x = conv_bn(x, 64)

    # ════════════════════════════════════════════════════════════════
    # ★ ATTENTION BLOCK 1: Local Channel Attention Block (LCAB)
    #   Recalibrates the 64 feature channels using global statistics.
    #   Channels encoding surface curvature / normal orientations are
    #   amplified; redundant or noisy channels are down-weighted.
    # ════════════════════════════════════════════════════════════════
    x = local_channel_attention_block(x, reduction_ratio=8, name='lcab')

    # ── Feature alignment ────────────────────────────────────────────
    x = tnet(x, 64)

    # ── Deep feature extraction ──────────────────────────────────────
    x = conv_bn(x, 64)
    x = conv_bn(x, 128)
    x = conv_bn(x, 1024)

    # ════════════════════════════════════════════════════════════════
    # ★ ATTENTION BLOCK 2: Global Self-Attention Block (GSAB)
    #   Multi-head self-attention across all N points so every point
    #   can incorporate context from every other point before the
    #   irreversible GlobalMaxPooling aggregation step.
    # ════════════════════════════════════════════════════════════════
    x = global_self_attention_block(x, num_heads=4, key_dim=64, name='gsab')

    # ── Global aggregation ───────────────────────────────────────────
    x = layers.GlobalMaxPooling1D(name='global_max_pool')(x)

    # ── Classification head ──────────────────────────────────────────
    x = dense_bn(x, 512)
    x = dense_bn(x, 256)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)

    return keras.Model(inputs=inputs, outputs=outputs,
                       name='AdaptivePointNet_DualAttention')

# Build and inspect
model = build_adaptivepointnet()
model.summary(line_length=110)

## 8. Train the Model

In [ ]:
def step_decay(epoch):
    """Step LR decay: starts at 0.001, halved every 20 epochs."""
    lrate = 0.001
    step  = 20
    if epoch > 20:
        lrate = lrate / (2 * (epoch // step))
    return lrate

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=keras.optimizers.Adam(learning_rate=0.0, beta_1=0.9, beta_2=0.999),
    metrics=['sparse_categorical_accuracy'],
)

callbacks = [
    keras.callbacks.LearningRateScheduler(step_decay, verbose=0),
    keras.callbacks.ModelCheckpoint(
        filepath=WEIGHTS_PATH,
        monitor='val_sparse_categorical_accuracy',
        mode='max', save_best_only=True, save_weights_only=True, verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_sparse_categorical_accuracy',
        patience=15, restore_best_weights=True, verbose=1,
    ),
]

history = model.fit(
    train_dataset,
    epochs=100,
    validation_data=test_dataset,
    callbacks=callbacks,
)

## 9. Learning Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

train_acc = history.history['sparse_categorical_accuracy']
val_acc   = history.history['val_sparse_categorical_accuracy']
train_loss= history.history['loss']
val_loss  = history.history['val_loss']

axes[0].plot(train_acc,  color='steelblue', lw=2, label='Train')
axes[0].plot(val_acc,    color='tomato',    lw=2, label='Validation')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].set_title('Classification Accuracy — AdaptivePointNet (Dual Attention)')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(train_loss, color='steelblue', lw=2, label='Train')
axes[1].plot(val_loss,   color='tomato',    lw=2, label='Validation')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Cross-Entropy Loss')
axes[1].set_title('Loss Curves')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Evaluation — Confusion Matrix

In [ ]:
# Load best checkpoint weights
model.load_weights(WEIGHTS_PATH)

preds   = model.predict(test_dataset)
preds   = tf.math.argmax(preds, -1).numpy()
val_acc = np.mean(test_labels == preds)
print(f'Validation accuracy: {val_acc:.4f}  ({val_acc*100:.2f}%)')

conf_matrix = confusion_matrix(test_labels, preds)
norm_matrix = conf_matrix / np.sum(conf_matrix, axis=1, keepdims=True)

plt.figure(figsize=(11, 9))
sns.set(font_scale=1.1)
ax = sns.heatmap(
    norm_matrix, annot=True, fmt='.1%', cmap='Blues',
    xticklabels=list(class_map.values()),
    yticklabels=list(class_map.values()),
)
ax.set_xlabel('Predicted Labels')
ax.set_ylabel('Ground Truth')
ax.set_title(f'Confusion Matrix — AdaptivePointNet Dual Attention\nVal Acc: {val_acc:.3%}')
plt.xticks(rotation=40, ha='right', fontsize=10)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Best & Worst Predicted Classes

In [ ]:
percent_sort = np.diag(norm_matrix)

best_3  = np.flip(np.argsort(percent_sort)[-3:])
worst_3 = np.argsort(percent_sort)[:3]

best_labels  = [class_map[v] for v in best_3]
worst_labels = [class_map[v] for v in worst_3]

fig, ax = plt.subplots(1, 2, figsize=(18, 5))
sns.barplot(y=best_labels,  x=percent_sort[best_3]*100,
            orient='h', color='teal',       ax=ax[0])
ax[0].xaxis.set_major_formatter(mtick.PercentFormatter())
ax[0].set_title('Top 3 Best Predicted Classes')

sns.barplot(y=worst_labels, x=percent_sort[worst_3]*100,
            orient='h', color='lightcoral', ax=ax[1])
ax[1].xaxis.set_major_formatter(mtick.PercentFormatter())
ax[1].set_title('Top 3 Worst Predicted Classes')

plt.suptitle('Per-Class Accuracy — AdaptivePointNet (Dual Attention)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('best_worst_classes.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Sample Predictions & Visualisations

In [ ]:
test_sample = tf.data.Dataset.from_tensor_slices((test_points, test_labels))
test_sample = test_sample.shuffle(len(test_points)).batch(BATCH_SIZE)

points, labels = list(test_sample.take(1))[0]
pts_np  = points[:8].numpy()
labs_np = labels[:8].numpy()
pred_8  = tf.math.argmax(model.predict(points[:8]), -1).numpy()

fig = plt.figure(figsize=(18, 9))
for i in range(8):
    ax = fig.add_subplot(2, 4, i + 1, projection='3d')
    ax.scatter(pts_np[i,:,0], pts_np[i,:,1], pts_np[i,:,2],
               s=1, c=pts_np[i,:,2], cmap='plasma')
    match = '✓' if pred_8[i] == labs_np[i] else '✗'
    ax.set_title(
        f'GT: {class_map[labs_np[i]]}\nPred: {class_map[pred_8[i]]} {match}',
        fontsize=9,
    )
    ax.set_axis_off()
plt.suptitle('Predictions — AdaptivePointNet (Dual Attention)', fontsize=13)
plt.tight_layout()
plt.savefig('predictions.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. t-SNE Visualisation of Global Features

In [ ]:
# Find the GlobalMaxPooling1D layer dynamically
pool_layer = next(l for l in model.layers if isinstance(l, layers.GlobalMaxPooling1D))
feature_model = keras.Model(inputs=model.input, outputs=pool_layer.output)
global_feat   = feature_model.predict(train_points, batch_size=32, verbose=1)

def scale(arr):
    rng = arr.max() - arr.min()
    return (arr - arr.min()) / (rng + 1e-8)

print('Fitting t-SNE (this may take a few minutes) …')
tsne = TSNE(
    n_components=2, perplexity=50, n_iter=5000,
    early_exaggeration=50, learning_rate='auto',
    n_iter_without_progress=300, verbose=1,
).fit_transform(global_feat)

tx = scale(tsne[:, 0])
ty = scale(tsne[:, 1])

fig = plt.figure(figsize=(20, 10))
ax  = fig.add_subplot(111)
for idx, c in enumerate(class_map.values()):
    mask = train_labels == idx
    ax.scatter(tx[mask], ty[mask], label=c, alpha=0.7, s=12)
    ax.annotate(c, (tx[mask].mean(), ty[mask].mean()),
                fontsize=13, fontweight='bold')
plt.title('t-SNE of 1024-d Global Features — AdaptivePointNet (Dual Attention)', fontsize=14)
plt.xlabel('Reduced Dim 1'); plt.ylabel('Reduced Dim 2')
plt.tight_layout()
plt.savefig('tsne.png', dpi=150, bbox_inches='tight')
plt.show()

## 14. Results Summary

In [ ]:
print("=" * 60)
print("  AdaptivePointNet — Dual Attention — Final Results")
print("=" * 60)
print(f"  Validation Accuracy : {val_acc:.4f}  ({val_acc*100:.2f}%)")
print(f"  Total Parameters    : {model.count_params():,}")
print()
print("  Novel contributions:")
print("  ★ LCAB (Local Channel Attention Block)")
print("    - Position : after first two conv_bn(64) layers")
print("    - Mechanism: Squeeze-and-Excitation channel recalibration")
print("    - Params   : 2×64×8 + 8×64 = 1,536")
print()
print("  ★ GSAB (Global Self-Attention Block)")
print("    - Position : after conv_bn(1024), before GlobalMaxPool")
print("    - Mechanism: Multi-Head Self-Attention (4 heads, key_dim=64)")
print("               + Feed-Forward Network (4096 → 1024)")
print("    - Params   : ~8.4M (dominated by FFN: 1024×4096 + 4096×1024)")
print("=" * 60)